# Instagram logging events from a HAR file

Instagram's web app sends batches of events to `graph.instagram.com/logging_client_events`. This notebook decodes them into a dataframe.

In [ ]:
import json
import urllib.parse

import pandas as pd

HAR_FILE = "www.instagram.com_Archive [26-11-12 14-29-33].har"

## 1. Decode every event

Each request's `message` field is JSON. Its `data` list holds the events.

In [ ]:
with open(HAR_FILE, encoding="utf-8") as f:
    har = json.load(f)

records = []
for entry in har["log"]["entries"]:
    request = entry["request"]
    body = request.get("postData", {}).get("text")
    if "logging_client_events" not in request["url"] or not body:
        continue
    message = json.loads(urllib.parse.parse_qs(body)["message"][0])
    records.extend(message["data"])

raw = pd.json_normalize(records)
raw.shape

## 2. Keep the useful fields

In [ ]:
columns = {
    "name": "event",
    "time": "timestamp",
    "extra.m_pk": "post_id",
    "extra.a_pk": "author_id",
    "extra.a_i": "ad_or_organic",
    "extra.follow_status": "follow_status",
    "extra.time": "position",          # seconds into the video
    "extra.lsp": "resumed_from",       # position this stretch of playback started at
    "extra.duration": "video_length",  # seconds
    "extra.loop_count": "loops",       # times the video has repeated
    "extra.reason": "reason",          # why playback started
    "extra.start_delay": "start_delay",
    "extra.c_pk": "comment_id",
    "extra.ca_pk": "comment_author_id",
    "extra.target_id": "suggested_account_id",
    "extra.position": "list_position",
    "extra.nav_chain": "screen",
}

events = raw.reindex(columns=list(columns)).rename(columns=columns)

events["timestamp"] = pd.to_datetime(events["timestamp"].astype(float), unit="s")
events["seconds_in"] = (events["timestamp"] - events["timestamp"].min()).dt.total_seconds()
# Last step of the navigation chain: feedPage (main feed) or postPage (a post opened in a window)
events["screen"] = events["screen"].str.split(",").str[-1].str.split(":").str[1]

events = events.sort_values("timestamp").reset_index(drop=True)
events["event"].value_counts()

## 3. Events by type

### `video_should_start`

A video was queued to play. `reason` says why (`autoplay`).

In [ ]:
events[events["event"] == "video_should_start"][['seconds_in', 'screen', 'post_id', 'ad_or_organic', 'reason']]

### `video_started_playing`

Playback actually began. `start_delay` is the wait before it did.

In [ ]:
events[events["event"] == "video_started_playing"][['seconds_in', 'screen', 'post_id', 'ad_or_organic', 'reason', 'start_delay']]

### `video_paused`

Playback stopped, by pausing or scrolling away. `position` is where it stopped; `resumed_from` is where that stretch began.

In [ ]:
events[events["event"] == "video_paused"][['seconds_in', 'screen', 'post_id', 'ad_or_organic', 'position', 'resumed_from', 'video_length', 'loops']]

### `video_exited`

The video left the screen. `position` and `loops` show how much had been watched.

In [ ]:
events[events["event"] == "video_exited"][['seconds_in', 'screen', 'post_id', 'ad_or_organic', 'position', 'video_length', 'loops']]

### `video_viewed_time`

A report of viewing time: `position` reached out of `video_length`.

In [ ]:
events[events["event"] == "video_viewed_time"][['seconds_in', 'screen', 'post_id', 'ad_or_organic', 'position', 'video_length', 'loops']]

### `video_audio_enabled`

The video was unmuted, at `position`.

In [ ]:
events[events["event"] == "video_audio_enabled"][['seconds_in', 'screen', 'post_id', 'ad_or_organic', 'position', 'video_length']]

### `video_audio_disabled`

The video was muted, at `position`.

In [ ]:
events[events["event"] == "video_audio_disabled"][['seconds_in', 'screen', 'post_id', 'ad_or_organic', 'position', 'video_length']]

### `video_buffering_started`

The video stalled to load (network, not the user).

In [ ]:
events[events["event"] == "video_buffering_started"][['seconds_in', 'screen', 'post_id', 'ad_or_organic', 'loops', 'follow_status']]

### `comment_impression`

A comment appeared on screen: which comment, on which post, by whom.

In [ ]:
events[events["event"] == "comment_impression"][['seconds_in', 'screen', 'post_id', 'comment_id', 'comment_author_id']]

### `recommended_user_impression`

A suggested account was shown, and where it sat in the list.

In [ ]:
events[events["event"] == "recommended_user_impression"][['seconds_in', 'screen', 'list_position', 'suggested_account_id']]

## 4. How much of each video was watched

Furthest point reached per video: `position` plus full loops.

In [ ]:
video = events[events["position"].notna() & (events["video_length"] > 0)].copy()
video["watched"] = video["position"] + video["loops"].fillna(0) * video["video_length"]

watched = video.groupby("post_id").agg(
    ad_or_organic=("ad_or_organic", "first"),
    video_length=("video_length", "max"),
    seconds_watched=("watched", "max"),
    first_seen=("seconds_in", "min"),
).sort_values("first_seen")
watched["percent_watched"] = (100 * watched["seconds_watched"] / watched["video_length"]).round(1)
watched